In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(vol.ravel(), bins=200, range=(-1100, 400), log=True)
plt.xlabel("Unités Hounsfield (HU)")
plt.ylabel("Nombre de voxels (échelle log)")
plt.title("Histogramme des HU")
plt.show()

In [ ]:
nod = ann[ann.seriesuid == uid].sort_values("diameter_mm", ascending=False)
print(nod)

n = nod.iloc[0]                                   # le plus gros nodule
centre_mm = np.array([n.coordX, n.coordY, n.coordZ])
vx, vy, vz = np.rint((centre_mm - origin) / spacing).astype(int)
rayon_px = n.diameter_mm / 2 / spacing[0]
print(f"Voxel (x,y,z) = ({vx}, {vy}, {vz}), diamètre {n.diameter_mm:.1f} mm")

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(vol[vz], cmap="gray", vmin=-1000, vmax=400)
axes[0].add_patch(plt.Circle((vx, vy), rayon_px + 10, color="red", fill=False, lw=2))
axes[0].set_title(f"coupe z = {vz}")

m = 32                                            # zoom 64 x 64 pixels
axes[1].imshow(vol[vz, vy-m:vy+m, vx-m:vx+m], cmap="gray", vmin=-1000, vmax=400)
axes[1].set_title("zoom sur le nodule")
for a in axes:
    a.axis("off")
plt.show()

In [ ]:
idx = np.linspace(vol.shape[0] * 0.2, vol.shape[0] * 0.8, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, k in zip(axes.ravel(), idx):
    ax.imshow(vol[k], cmap="gray", vmin=-1000, vmax=400)
    ax.set_title(f"coupe z = {k}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
row = patients[patients.split == "dev"].iloc[0]
uid, subset = row.seriesuid, row.subset
path = next((LUNA / subset).rglob(f"{uid}.mhd"))

img = sitk.ReadImage(str(path))
vol = sitk.GetArrayFromImage(img)        # ordre (z, y, x)
spacing = np.array(img.GetSpacing())     # (x, y, z) en mm
origin = np.array(img.GetOrigin())       # (x, y, z) en mm

print("Patient     :", uid)
print("Forme (z,y,x):", vol.shape)
print("Espacement (x,y,z) mm:", spacing)
print("HU min / max:", vol.min(), vol.max())

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import SimpleITK as sitk

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
LUNA = ROOT / "data" / "luna16"

patients = pd.read_csv(ROOT / "results" / "patients.csv")
ann = pd.read_csv(LUNA / "annotations.csv")
patients.head()